Develop an Information Retrieval System for any domain
The goal of this assignment is to develop search engines for any specific domain (music, health care, education, etc.). The search engine development should include
- Pre-processing of data
- Building indexes
- Compression of indexes
- IR- models – VSM, BM 25
- Query mapping
- Display of Results (top k results)
- Explainable results

For example, build a search engine for searching song lyrics.
Dataset: https://labrosa.ee.columbia.edu/millionsong/musixmatch

In [1]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
# You need to specify a file within the dataset with a supported extension (e.g., .csv, .json).
# For example, let's assume there is a CSV file named 'msd_genre_dataset.csv'.
# You might need to change this based on the actual file names in the dataset.
file_path = "spotify_millsongdata.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "notshrirang/spotify-million-song-dataset",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:")
display(df.head())

/tmp/ipykernel_738/443079173.py:13: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


100%|██████████| 20.7M/20.7M [00:00<00:00, 134MB/s] 

Extracting zip of spotify_millsongdata.csv...


First 5 records:


,artist,song,link,text
0,ABBA,Ahe's My Kind Of Girl,/a/abba/ahes+my+kind+of+girl_20598417.html,"Look at her face, it's a wonderful face \r\nA..."
1,ABBA,"Andante, Andante",/a/abba/andante+andante_20002708.html,"Take it easy with me, please \r\nTouch me gen..."
2,ABBA,As Good As New,/a/abba/as+good+as+new_20003033.html,I'll never know why I had to go \r\nWhy I had...
3,ABBA,Bang,/a/abba/bang_20598415.html,Making somebody happy is a question of give an...
4,ABBA,Bang-A-Boomerang,/a/abba/bang+a+boomerang_20002668.html,Making somebody happy is a question of give an...


In [ ]:
pip install nltk

In [ ]:
import pandas as pd
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

nltk.download('stopwords')
nltk.download('punkt_tab')

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_lyrics(text):
  # a. Handle empty values
  if not isinstance(text, str):
      return []

  # b. Lowercasing & Newline removal
  text = text.lower().replace('\n', ' ').replace('\r', ' ')

  # c. Remove punctuation and special characters
  text = re.sub(r'[^a-z0-9\s]', '', text)

  # d. Tokenization
  tokens = word_tokenize(text)

  # e. Stopword Removal and Stemming
  processed_tokens = [
      stemmer.stem(word)
      for word in tokens
      if word not in stop_words and len(word) > 2
  ]

  return processed_tokens


# Apply to a subset - Let's take 10,000 songs for development speed
df_sample = df.head(10000).copy()

# *** Pre-processing the lyrics ***
print("Starting pre-processing... This may take a minute.")
df_sample['processed_tokens'] = df_sample['text'].apply(preprocess_lyrics)
print("Pre-processing complete!")

# View the transformation
print("\n--- Comparison ---")
print(f"Original Lyrics Snippet: {df_sample['text'].iloc[0][:50]}...")
print(f"Processed Tokens: {df_sample['processed_tokens'].iloc[0][:10]}")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Starting pre-processing... This may take a minute.
Pre-processing complete!

--- Comparison ---
Original Lyrics Snippet: Look at her face, it's a wonderful face  
And it ...
Processed Tokens: ['look', 'face', 'wonder', 'face', 'mean', 'someth', 'special', 'look', 'way', 'smile']


In [ ]:
# Why Stemming?
# If a user searches for "crying", the stemmer converts it to "cri". In our index, the song lyrics with "cried" will also be stored as "cri".
# This creates a match that wouldn't happen with raw text.
#
# Why len(word) > 2?
# In lyrics, there are many "oh", "ah", "la" sounds. Filtering out very short words helps focus the search engine on meaningful content.

In [ ]:
from collections import defaultdict

# *** Build the Inverted Index ***
print("Building Inverted Index...")
inverted_index = defaultdict(list)

for idx, tokens in enumerate(df_sample['processed_tokens']):
    for token in set(tokens):
        inverted_index[token].append(idx)


items = list(inverted_index.items())
print(items[:2])

Building Inverted Index...
[('wonder', [0, 12, 21, 37, 40, 41, 63, 76, 81, 82, 83, 91, 95, 114, 123, 130, 137, 140, 164, 170, 179, 184, 201, 232, 233, 246, 252, 267, 277, 326, 391, 407, 444, 493, 502, 511, 519, 567, 578, 581, 588, 591, 605, 610, 613, 614, 617, 621, 627, 628, 630, 648, 650, 657, 679, 691, 703, 704, 705, 713, 727, 735, 747, 783, 805, 836, 847, 849, 887, 916, 926, 950, 966, 970, 982, 986, 1000, 1028, 1038, 1041, 1051, 1060, 1078, 1080, 1086, 1096, 1104, 1113, 1115, 1132, 1139, 1144, 1154, 1184, 1189, 1202, 1236, 1240, 1258, 1267, 1269, 1272, 1288, 1330, 1339, 1387, 1391, 1407, 1431, 1433, 1448, 1467, 1486, 1489, 1520, 1540, 1541, 1580, 1581, 1596, 1598, 1612, 1648, 1660, 1665, 1672, 1673, 1683, 1699, 1710, 1715, 1737, 1740, 1745, 1750, 1769, 1775, 1844, 1848, 1860, 1865, 1878, 1879, 1885, 1898, 1905, 1907, 1912, 1915, 1928, 1934, 1937, 1949, 1953, 1965, 1997, 2022, 2073, 2079, 2087, 2089, 2136, 2144, 2225, 2241, 2281, 2283, 2293, 2306, 2343, 2352, 2369, 2393, 2395, 2420, 

In [ ]:
# *** Apply Compression ***

def vb_encode_number(number):
  """Encodes a single number into Variable Byte format."""
  bytes_list = []
  while True:
      bytes_list.insert(0, number % 128)
      if number < 128:
          break
      number //= 128
  bytes_list[-1] += 128
  return bytes(bytes_list)

def vb_encode_list(numbers):
  """Encodes a list of numbers (gaps) into a stream of bytes."""
  return b"".join([vb_encode_number(n) for n in numbers])



print("Compressing Index using Delta and VB Encoding...")
compressed_index = {}

for term, doc_ids in inverted_index.items():
  # Sort IDs to calculate gaps
  doc_ids.sort()

  # Delta Encoding: [100, 105, 110] -> [100, 5, 5]
  gaps = [doc_ids[0]] + [doc_ids[i] - doc_ids[i-1] for i in range(1, len(doc_ids))]

  # Variable Byte Compression
  compressed_index[term] = vb_encode_list(gaps)


print(f"Index built! Total unique terms: {len(compressed_index)}")

example_term = list(compressed_index.keys())[0]
print(f"\nTerm: '{example_term}'")
print(f"Compressed bytes: {compressed_index[example_term][:10]}...")

Compressing Index using Delta and VB Encoding...
Index built! Total unique terms: 26536

Term: 'wonder'
Compressed bytes: b'\x80\x8c\x89\x90\x83\x81\x96\x8d\x85\x81'...


In [ ]:
# *** IR-Models ***

import math
import collections

# Total number of songs
N = len(df_sample)

# Average length of lyrics (needed for BM25)
# This helps the model realize that 5 matches in a short song
# is more impressive than 5 matches in a very long song.
avg_dl = df_sample['processed_tokens'].map(len).mean()

print(f"Total Songs (N): {N}")
print(f"Average Song Length: {avg_dl:.2f} words")

Total Songs (N): 10000
Average Song Length: 107.43 words


In [ ]:
pip install rank_bm25

In [ ]:
from rank_bm25 import BM25Okapi

# This prepares the library model
tokenized_corpus = df_sample['processed_tokens'].tolist()
bm25_library_engine = BM25Okapi(tokenized_corpus)

In [ ]:
def get_scores(query, model='bm25'):
    query_tokens = preprocess_lyrics(query)
    scores = collections.defaultdict(float)

    if model == 'bm25':
        # --- Using the In-built BM25 ---
        # This returns an array of scores for all documents in the corpus
        library_scores = bm25_library_engine.get_scores(query_tokens)

        for idx, score in enumerate(library_scores):
            if score > 0:
                scores[idx] = score

        """
        # --- MANUAL BM25 FORMULA ---
        k1, b = 1.5, 0.75
        for token in query_tokens:
            if token in inverted_index:
                df_t = len(inverted_index[token])
                idf = math.log((N - df_t + 0.5) / (df_t + 0.5) + 1)
                for doc_idx in inverted_index[token]:
                    tf = df_sample['processed_tokens'].iloc[doc_idx].count(token)
                    dl = len(df_sample['processed_tokens'].iloc[doc_idx])
                    numerator = tf * (k1 + 1)
                    denominator = tf + k1 * (1 - b + b * (dl / avg_dl))
                    scores[doc_idx] += idf * (numerator / denominator)
        """

    elif model == 'vsm':
        for token in query_tokens:
            if token in inverted_index:
                df_t = len(inverted_index[token])
                idf = math.log10(N / df_t)
                for doc_idx in inverted_index[token]:
                    tf = df_sample['processed_tokens'].iloc[doc_idx].count(token)
                    # TF-IDF calculation
                    # --- VSM Formula ---
                    # Score = (1 + log(tf)) * idf
                    tf_weight = (1 + math.log10(tf))
                    scores[doc_idx] += tf_weight * idf

    return scores, query_tokens

In [ ]:
def search_and_display(query, model='bm25', k=3):
    scores, query_tokens = get_scores(query, model=model)

    # Sort results
    sorted_results = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]

    print(f"\nSEARCH RESULTS: '{query}' ({model.upper()})")
    print("="*65)

    if not sorted_results:
        print("No matches found.")
        return

    for rank, (doc_idx, score) in enumerate(sorted_results, 1):
        row = df_sample.iloc[doc_idx]
        print(f"Rank {rank}: {row['song']} - {row['artist']}")
        print(f"Relevance Score: {score:.4f}")

        # *** Explainable Results ***
        # Identifying which tokens triggered the result
        matches = [t for t in query_tokens if t in row['processed_tokens']]
        print(f"Explanation: Matches found for stems {matches}.")

        snippet = row['text'].replace('\n', ' ')[:110]
        print(f"Snippet: {snippet}...")
        print("-" * 65)


# --- EXECUTION ---
search_and_display("kill my heart", model='bm25', k=1)


SEARCH RESULTS: 'kill my heart' (BM25)
Rank 1: Dressed To Kill - Cher
Relevance Score: 9.3803
Explanation: Matches found for stems ['kill', 'heart'].
 You know it's gonna ...
-----------------------------------------------------------------


### Without using Libraries and included BIM

In [2]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
# You need to specify a file within the dataset with a supported extension (e.g., .csv, .json).
# For example, let's assume there is a CSV file named 'msd_genre_dataset.csv'.
# You might need to change this based on the actual file names in the dataset.
file_path = "spotify_millsongdata.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "notshrirang/spotify-million-song-dataset",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:")
display(df.head())

/tmp/ipykernel_738/443079173.py:13: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'spotify-million-song-dataset' dataset.
First 5 records:


,artist,song,link,text
0,ABBA,Ahe's My Kind Of Girl,/a/abba/ahes+my+kind+of+girl_20598417.html,"Look at her face, it's a wonderful face \r\nA..."
1,ABBA,"Andante, Andante",/a/abba/andante+andante_20002708.html,"Take it easy with me, please \r\nTouch me gen..."
2,ABBA,As Good As New,/a/abba/as+good+as+new_20003033.html,I'll never know why I had to go \r\nWhy I had...
3,ABBA,Bang,/a/abba/bang_20598415.html,Making somebody happy is a question of give an...
4,ABBA,Bang-A-Boomerang,/a/abba/bang+a+boomerang_20002668.html,Making somebody happy is a question of give an...


In [8]:
import re
from collections import defaultdict

stopwords = ["is","was","are","were","the","a","am","and","or","for","it","that","there"]

def preprocess_lyrics(text):
  text = text.lower().strip()
  text = re.sub(r'[^a-z0-9\s]', '', text)
  words = text.split()
  filtered_words = [word for word in words if word not in stopwords]
  return filtered_words

def vb_encode_number(number):
  byte_list = []
  while True:
    byte_list.insert(0, number%128)
    if number < 128: break
    number //= 128
  byte_list[-1] += 128
  return bytes(byte_list)

def vb_encode_list(numbers):
  return b"".join([vb_encode_number(n) for n in numbers])



df_sample = df.head(10000).copy()

print("Starting pre-processing... This may take a minute.")
df_sample['processed_tokens'] = df_sample['text'].apply(preprocess_lyrics)
print("Pre-processing complete!")

# Build indices
inverted_index = defaultdict(list)
for idx, words in enumerate(df_sample['processed_tokens']):
    for word in set(words):
        inverted_index[word].append(idx)
print("Built inverted index!")

# Compression of index
compressed_index = {}
for term, doc_id in inverted_index.items():
    doc_id.sort()
    n = len(doc_id)
    gaps = [doc_id[0]] + [doc_id[i] - doc_id[i-1] for i in range(1,n)]
    compressed_index[term] = vb_encode_list(gaps)
print("Index compression done!")

N = len(df_sample['processed_tokens'])
avg_dl = df_sample['processed_tokens'].map(len).mean()

Starting pre-processing... This may take a minute.
Pre-processing complete!
Built inverted index!
Index compression done!


In [10]:
def get_cosine_similarity(vec1, vec2):
  numerator = sum([vec1[x] * vec2[x] for x in vec1.keys()])
  sum1 = sum([vec1[x]**2 for x in vec1.keys()])
  sum2 = sum([vec2[x]**2 for x in vec2.keys()])
  denominator = math.sqrt(sum1) * math.sqrt(sum2)
  if denominator == 0:
    return 0
  return numerator / denominator

In [13]:
from collections import defaultdict
import math

def get_scores_VSM_model(query, inverted_index):
    query_tokens = preprocess_lyrics(query)
    scores = defaultdict(float)

    for token in query_tokens:
        if token in inverted_index:
            df_t = len(inverted_index[token])
            idf = math.log10(N / df_t)
            for doc_idx in inverted_index[token]:
                tf = df_sample['processed_tokens'].iloc[doc_idx].count(token)
                tf_weight = (1 + math.log10(tf))
                scores[doc_idx] += tf_weight * idf
    return scores, query_tokens

    '''
    USING COSINE SIMILARITY

    # 1. Build Query TF-IDF vector
    query_vector = {}
    for token in set(query_tokens):
        if token in inverted_index:
            df_t = len(inverted_index[token])
            IDF = math.log10(N / df_t)
            tf = query_tokens.count(token)
            tf_weight = 1 + math.log10(tf)
            query_vector[token] = tf_weight * IDF

    # 2. Build Document vectors
    for doc_idx in range(N):
        doc_vector = {}
        for token in query_vector:
            tf = df_sample['processed_tokens'].iloc[doc_idx].count(token)
            if tf > 0:
                df_t = len(inverted_index[token])
                IDF = math.log10(N / df_t)
                tf_weight = 1 + math.log10(tf)
                doc_vector[token] = tf_weight * IDF
            else:
                doc_vector[token] = 0

        # 3. Cosine similarity
        scores[doc_idx] = get_cosine_similarity(query_vector,doc_vector)

    return scores, query_tokens
    '''

In [14]:
def get_scores_BM25_model(query, inverted_index):
    query_tokens = preprocess_lyrics(query)
    scores = defaultdict(float)

    k, b = 1.5, 0.75
    for token in query_tokens:
        if token in inverted_index:
            df_t = len(inverted_index[token])
            idf = math.log10((N - df_t + 0.5)/(1 + df_t + 0.5))
            for doc_idx in inverted_index[token]:
                tf = df_sample['processed_tokens'].iloc[doc_idx].count(token)
                dl = len(df_sample['processed_tokens'].iloc[doc_idx])
                numerator = tf * (k + 1)
                denominator = tf + k * (1 - b + b * (dl / avg_dl))
                scores[doc_idx] += idf * (numerator / denominator)

    return scores, query_tokens

In [23]:
def get_scores_BIM_model(query, inverted_index, relevant_doc_ids):
    query_tokens = preprocess_lyrics(query)
    scores = defaultdict(float)

    Nr = len(relevant_doc_ids)
    Nd = len(df_sample['processed_tokens'])

    for token in query_tokens:
        if token in inverted_index:
            dk = len(inverted_index[token])
            rk = 0
            for doc_idx in relevant_doc_ids:
                if token in df_sample['processed_tokens'].iloc[doc_idx]:
                    rk += 1
            pk = (rk + 0.5) / (Nr+1)
            qk = (dk - rk + 0.5) / (Nd - Nr + 1)
            tr_k = math.log((pk/(1-pk)) / (qk/(1-qk)))
            for doc_idx in inverted_index[token]:
                scores[doc_idx] += tr_k

    return scores, query_tokens

In [27]:
def get_scores(model, query):
  if model=='vsm':
    return get_scores_VSM_model(query, inverted_index)
  elif model=='bm25':
    return get_scores_BM25_model(query, inverted_index)
  elif model=='bim':
    return get_scores_BIM_model(query, inverted_index, relevant_doc_ids=[0,1,2])

def search_and_display(query, model, k=3):
    scores, query_tokens = get_scores(model,query)

    sorted_results = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]

    print(f"\nSEARCH RESULTS: '{query}' ({model.upper()})")
    print("="*65)

    if not sorted_results:
        print("No matches found.")
        return

    for rank, (doc_idx, score) in enumerate(sorted_results, 1):
        row = df_sample.iloc[doc_idx]
        print(f"Rank {rank}: {row['song']} - {row['artist']}")
        print(f"Relevance Score: {score:.4f}")

        matches = [t for t in query_tokens if t in row['processed_tokens']]
        print(f"Explanation: Matches found for - {matches}")

        snippet = row['text'].replace('\n', ' ')[:110]
        print(f"Snippet: {snippet}...")
        print("-" * 65)


# --- EXECUTION ---
# search_and_display("kill my heart", model='bm25', k=1)
# search_and_display("heart", model='bim', k=3)
search_and_display("heart", model='vsm', k=5)



SEARCH RESULTS: 'heart' (VSM)
Rank 1: Don't Go Breaking My Heart - Elton John
Relevance Score: 1.6432
Explanation: Matches found for - ['heart']
 Baby you're not that kind...
-----------------------------------------------------------------
Rank 2: Bury My Heart At Wounded Knee - Indigo Girls
Relevance Score: 1.6241
Explanation: Matches found for - ['heart']
 And he don't know much about the issues so he pi...
-----------------------------------------------------------------
Rank 3: Piece Of My Heart - Faith Hill
Relevance Score: 1.5577
Explanation: Matches found for - ['heart']
 That a woman possi...
-----------------------------------------------------------------
Rank 4: Heart Of Gold - Boney M.
Relevance Score: 1.5450
Explanation: Matches found for - ['heart']
 All these expressions I never gi...
-----------------------------------------------------------------
Rank 5: Invaders Of The Heart - Cheap Trick
Relevance Score: 1.5450
Explanation: Matches found for - ['heart']
 Are messi